# Turtle Breakout 13 — the frozen specification on untouched data: new coins (2020–2026) and spot 2017–2019

Pre-registration: `discussions/Turtle_Breakout/015_claude_tb13_preregistration.md`. Code: `turtle_oos.py`, `turtle_portfolio.py`; data: `tools/download_tb13_data.py`.

**Frozen spec (TB 12):** one account; TB 01 daily longs + TB 07 reverse short at each losing stop exit; no portfolio rule; risk per trade $f \times$ current equity, primary $f = 0.5\%$.
**NEW:** USD-M perpetuals never used in TB 04–12, chosen by rule (delisted included). **SPOT:** Binance spot USDT pairs, 2017-08 → 2019-12 (funding 0).
**Gates per set** (Bonferroni over 4, $\alpha = 0.0125$ one-sided): A growth of long + reverse > 0; B Calmar(long + reverse) − Calmar(long) > 0.

Environment: `TB13_DATA_DIR` (default `<BINANCE_UM_DIR>\tb13`), `TB13_SYNTHETIC=1`.

In [ ]:
import os, sys, json, gc, time, subprocess, tempfile, warnings, platform
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_trade_features as tf, turtle_sizing as ts, turtle_reverse as tr, turtle_portfolio as tp, turtle_oos as to

SYNTHETIC = os.getenv('TB13_SYNTHETIC', '0') == '1'
OP = to.OParams(); P = tb.Params(); RP = tr.RParams()
PP = replace(tp.PParams(), size_base='equity', status='PREREGISTERED_TB015')
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb13-')) / '13'; OP = to.OParams(n_boot=500)
    U = {'new': {'chosen': [f'N{i}USDT' for i in range(6)]}, 'spot': {'chosen': [f'S{i}USDT' for i in range(4)]}}
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '13'
    U = json.loads((OUT / '13_universe.json').read_text())
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB13_DATA_DIR', str(base / 'tb13')))
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
for t in ('test_turtle_oos.py', 'test_turtle_portfolio.py'):
    chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', t], cwd=str(ROOT), env=env, capture_output=True, text=True)
    print(t, chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0
print(json.dumps(to.registration(OP), indent=1, default=str)[:1500])
print('NEW chosen:', len(U['new']['chosen']), '| SPOT chosen:', len(U['spot']['chosen']))
if not SYNTHETIC:
    print({k: U[k].get('eligible') for k in ('new', 'spot')}, U['new']['chosen'], U['spot']['chosen'])

## 1. Books per coin (costs as TB 12; SPOT also with spot fees and with an assumed funding rate)

In [ ]:
def books_for(sym, b, fund, start, cost_side=None):
    p = P if cost_side is None else replace(P, fee=cost_side - P.slip)
    rp = RP if cost_side is None else replace(RP, side_cost=cost_side)
    D = tb.prepare_asset(sym, b, fund, p)
    L0 = ts.trade_list(D, p, start, rp, with_reverse=False); L1 = ts.trade_list(D, p, start, rp, with_reverse=True)
    return tp.make_book(D, L0, p), tp.make_book(D, L1, p), D

SETS = {}
for kind in ('new', 'spot'):
    books = {'long': {}, 'long_rev': {}}; sens = {}; closes = {}; cov = []
    if kind == 'spot':
        sens = {'cost_high': {'long': {}, 'long_rev': {}}, 'funding': {'long': {}, 'long_rev': {}}}
    for i, sym in enumerate(U[kind]['chosen']):
        t1 = time.time()
        if SYNTHETIC:
            b = tb.synthetic_5m(900 if kind == 'new' else 800, seed=300 + i + (50 if kind == 'spot' else 0), trend=0.01,
                                start='2021-01-01' if kind == 'new' else '2017-09-01'); fund = tb.synthetic_funding(b, 0.0001) if kind == 'new' else None
        else:
            p1 = DATA / ('um' if kind == 'new' else 'spot') / f"{sym}_{'um' if kind == 'new' else 'spot'}_1m.parquet"
            if not p1.exists():
                cov.append({'symbol': sym, 'status': 'missing'}); print(sym, 'missing'); continue
            b, _ = tf.load_um(p1)
            last = pd.Timestamp(U[kind]['windows'][sym][1]) + pd.Timedelta(days=1)
            last = last.tz_localize('UTC') if last.tzinfo is None else last
            b = b[b.index <= last]
            if kind == 'spot':
                b = b[b.index <= pd.Timestamp(OP.spot_end, tz='UTC') + pd.Timedelta(days=1)]
            fp = DATA / 'funding' / f'{sym}_funding.parquet'
            fund = tb.funding_events(pd.read_parquet(fp)) if (kind == 'new' and fp.exists()) else None
        if len(b) < 288 * 60:
            cov.append({'symbol': sym, 'status': 'too short'}); continue
        start = b.index[0] + pd.Timedelta(days=30)
        bl, br, D = books_for(sym, b, fund, start)
        books['long'][sym], books['long_rev'][sym] = bl, br
        closes[sym] = pd.Series(bl.close, index=tb.period_end(bl.pid0 + np.arange(len(bl.close)), 'day'))
        if kind == 'spot':
            hl, hr, _ = books_for(sym, b, None, start, cost_side=OP.spot_cost_high)
            sens['cost_high']['long'][sym], sens['cost_high']['long_rev'][sym] = hl, hr
            fl, fr, _ = books_for(sym, b, tb.synthetic_funding(b, OP.spot_funding_sens), start)
            sens['funding']['long'][sym], sens['funding']['long_rev'][sym] = fl, fr
        cov.append({'symbol': sym, 'status': 'ok', 'start': start, 'end': b.index[-1], 'longs': len(bl.trades),
                    'reverses': int((br.trades.kind == 'reverse').sum()), 'sum_R_long': float(bl.trades.R.sum()),
                    'sum_R_reverse': float(br.trades.loc[br.trades.kind == 'reverse', 'R'].sum()), 'funding': fund is not None})
        print(f'{kind} {sym}: {len(bl.trades)} longs, {cov[-1]["reverses"]} reverses, {time.time() - t1:.0f}s')
        del b, D; gc.collect()
    SETS[kind] = {'books': books, 'sens': sens, 'closes': closes, 'cov': pd.DataFrame(cov)}
    display(SETS[kind]['cov'].round(2))

## 2. One account, frozen spec, both sets

In [ ]:
def run(books, f_, t0, t1):
    o = tp.simulate(books, 'R0', replace(PP, f=f_), t0, t1)
    D = tp.daily_curve(o['equity'], PP.start_equity)
    return D, {**tp.metrics(D, PP.start_equity), 'max_open_risk': o['max_open_risk'], 'mean_open_risk': o['mean_open_risk'],
               'max_gross_lev': o['max_gross_lev'], **o['stats']}
RES, CUR = [], {}
for kind in ('new', 'spot'):
    S = SETS[kind]; t0 = min(b.trades.t_in.min() for b in S['books']['long'].values()) - pd.Timedelta(minutes=5)
    t1 = None if kind == 'new' else pd.Timestamp(OP.spot_end, tz='UTC') + pd.Timedelta(hours=23)
    S['t0'], S['t1'] = t0, t1
    for f_ in OP.f_report:
        for v in ('long', 'long_rev'):
            D, m = run(S['books'][v], f_, t0, t1); CUR[(kind, f_, v)] = D
            RES.append({'set': kind, 'f': f_, 'variant': v, 'case': 'main', **m})
    if kind == 'spot':
        for case in ('cost_high', 'funding'):
            for v in ('long', 'long_rev'):
                D, m = run(S['sens'][case][v], OP.f_primary, t0, t1); CUR[(kind, case, v)] = D
                RES.append({'set': kind, 'f': OP.f_primary, 'variant': v, 'case': case, **m})
RES = pd.DataFrame(RES)
display(RES[['set', 'f', 'variant', 'case', 'cagr', 'max_dd', 'calmar', 'sharpe', 'ruined', 'max_open_risk', 'mean_open_risk', 'max_gross_lev', 'entries']].round(3))

## 3. Decision

In [ ]:
DEC = {}
for j, kind in enumerate(('new', 'spot')):
    A = to.gate_growth(CUR[(kind, OP.f_primary, 'long_rev')], PP.start_equity, OP, OP.seed + j)
    B = to.gate_calmar(CUR[(kind, OP.f_primary, 'long_rev')], CUR[(kind, OP.f_primary, 'long')], PP.start_equity, OP, OP.seed + 10 + j)
    DEC[kind] = {'A_growth': A, 'B_reversal': B, 'verdict': to.verdict(A['pass'], B['pass'])}
print(json.dumps(DEC, indent=1, default=float))

## 4. Reported: benchmarks, matched exposure, by year, per coin, sensitivities

In [ ]:
rows = []
for kind in ('new', 'spot'):
    S = SETS[kind]; lr = to.equal_weight_hold(S['closes'])
    lr = lr[(lr.index >= S['t0']) & ((lr.index <= S['t1']) if S['t1'] is not None else True)]
    D = PP.start_equity * np.exp(lr.cumsum()); CUR[(kind, 'hold')] = D
    rows.append({'set': kind, 'benchmark': 'equal-weight buy and hold (daily rebalanced)', **tp.metrics(D, PP.start_equity)})
    # long at the exposure of long + reverse (reversal effect net of exposure)
    e = RES[(RES.set == kind) & (RES.case == 'main') & (RES.f == OP.f_primary)].set_index('variant')
    fm = OP.f_primary * e.loc['long_rev', 'mean_open_risk'] / e.loc['long', 'mean_open_risk']
    D2, m2 = run(S['books']['long'], fm, S['t0'], S['t1']); CUR[(kind, 'long_matched')] = D2
    rows.append({'set': kind, 'benchmark': f'long only at f = {fm:.3%} (exposure of long + reverse)', **m2})
    rows[-1]['calmar_diff_vs_long_rev'] = to.gate_calmar(CUR[(kind, OP.f_primary, 'long_rev')], D2, PP.start_equity, OP, OP.seed + 20)['calmar_diff']
BENCH = pd.DataFrame(rows); display(BENCH.round(3))

def yearly(D):
    y = D.groupby(D.index.year).last(); prev = pd.concat([pd.Series([PP.start_equity], index=[y.index[0] - 1]), y]).shift(1).iloc[1:]
    return (y / prev - 1)
YR = pd.concat({f'{k}|{v}': yearly(CUR[(k, OP.f_primary, v)]) for k in ('new', 'spot') for v in ('long', 'long_rev')}, axis=1)
YR = pd.concat([YR, pd.concat({f'{k}|hold': yearly(CUR[(k, 'hold')]) for k in ('new', 'spot')}, axis=1)], axis=1)
print('Calendar-year returns, f = 0.5%'); display(YR.round(3))
for kind in ('new', 'spot'):
    c = SETS[kind]['cov']
    if len(c) and 'sum_R_long' in c:
        ok = c[c.status == 'ok']
        print(f"{kind}: coins {len(ok)} | mean R sum per coin: long {ok.sum_R_long.mean():.2f}, reverse {ok.sum_R_reverse.mean():.2f} | "
              f"coins with positive long R {int((ok.sum_R_long > 0).sum())}, positive reverse R {int((ok.sum_R_reverse > 0).sum())}")

## 5. Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for j, kind in enumerate(('new', 'spot')):
    for key, col, lab in (((kind, OP.f_primary, 'long'), '#1f5aa6', 'long'), ((kind, OP.f_primary, 'long_rev'), '#b33f40', 'long + reverse'),
                          ((kind, 'long_matched'), '#1f5aa6', 'long, matched exposure'), ((kind, 'hold'), '0.5', 'equal-weight hold')):
        c = CUR[key]; ls = ':' if 'matched' in lab else ('--' if 'hold' in lab else '-')
        axes[0, j].plot(c.index, c / PP.start_equity, color=col, ls=ls, lw=1.3, label=lab)
        axes[1, j].plot(c.index, (c / c.cummax() - 1) * 100, color=col, ls=ls, lw=1.0, label=lab)
    axes[0, j].set_yscale('log'); axes[0, j].set_title(f'{kind.upper()}: equity (f = 0.5%)', loc='left', fontsize=10)
    axes[1, j].set_title(f'{kind.upper()}: drawdown %', loc='left', fontsize=10)
    for a in axes[:, j]:
        a.grid(alpha=0.25); a.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 6. Save

In [ ]:
RES.to_csv(OUT / '13_results.csv', index=False); BENCH.to_csv(OUT / '13_benchmarks.csv', index=False); YR.to_csv(OUT / '13_yearly.csv')
for kind in ('new', 'spot'):
    SETS[kind]['cov'].to_csv(OUT / f'13_coverage_{kind}.csv', index=False)
pd.DataFrame({'|'.join(map(str, k)): c for k, c in CUR.items()}).to_csv(OUT / '13_curves.csv.gz', compression='gzip')
(OUT / '13_decision.json').write_text(json.dumps({'decision': DEC, 'registration': to.registration(OP), 'synthetic': SYNTHETIC}, indent=1, default=str))
git = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=str(ROOT), capture_output=True, text=True).stdout.strip()
(OUT / '13_manifest.json').write_text(json.dumps({'version': to.VERSION, 'portfolio_version': tp.VERSION, 'git': git, 'python': platform.python_version(),
    'created_utc': pd.Timestamp.now('UTC').isoformat(), 'synthetic': SYNTHETIC}, indent=1))
print('saved to', OUT); print({k: v['verdict'] for k, v in DEC.items()})